# Galerie explicabilité — 01 · Expliquer une décision, pas à pas 🟢

> Étagère asynchrone **optionnelle**. Pas un brief, pas de livrable, pas de note.
>
> ⏱️ ~2 h · 🟢 **entièrement résolu** : tout tourne, tu lis, tu relances, tu modifies.
> Le notebook 02 te demandera de refaire le geste **sans guidage**, sur un modèle à toi.

## Pourquoi ce notebook

Depuis M4, tu notes l'**explicabilité** dans la grille de décision C4
(`grille_decision_C4.md`, axe 3) et à l'étape 5 de la feuille de route du cas
d'usage. Tu la notes… sans jamais avoir produit une explication. C'est ce trou
qu'on comble ici.

Le contexte : **Crédit Aubrac**, banque régionale, mission FastIA. Ils ont un
modèle de risque qui refuse des dossiers, et une obligation très concrète —
dire au client **pourquoi** son dossier est refusé.

À la fin, tu sauras :

1. reconnaître quand l'explicabilité est **gratuite** (et donc quand SHAP est du sur-engineering) ;
2. produire une **importance globale** honnête avec scikit-learn seul ;
3. produire une **explication locale** SHAP, et la traduire en une phrase pour le client ;
4. utiliser l'explicabilité comme **détecteur de variable proxy** (passerelle C2 / M2-B2) ;
5. situer LIME face à SHAP, et savoir ce que **ni l'un ni l'autre** ne te dit.

**Fiches à garder ouvertes** : `fiche_interpretabilite_xai.pdf`,
`grille_decision_C4.md`, `fiche_pattern_ML_supervise.md`.

## § 0 — Les définitions, d'abord

Quatre mots à ne plus jamais confondre. Relis-les une fois, ils structurent tout
le notebook.

- **Explication globale** — *comment le modèle se comporte en général*, sur
  l'ensemble des dossiers. Répond à : « sur quoi ce modèle s'appuie-t-il ? ».
  C'est ce qu'on met dans une note de conception ou une model card.
- **Explication locale** — *pourquoi CE dossier a reçu CETTE prédiction*.
  Répond à : « pourquoi M. Untel est-il refusé ? ». C'est ce qu'exige un client,
  un juriste, un régulateur.
- **Interprétable par construction** — le modèle **est** son explication :
  régression linéaire/logistique, arbre de faible profondeur, système de règles.
  Rien à installer, rien à approximer.
- **Post-hoc** — le modèle est opaque, on lui **applique après coup** une méthode
  d'explication (SHAP, LIME, importance par permutation). C'est une
  **approximation**, avec ses hypothèses et ses angles morts.

> ⚠️ **Le piège dont tout le reste découle** : une explication dit ce que le
> **modèle** utilise, **jamais** ce qui *cause* le défaut dans le monde réel.
> « Le modèle s'appuie beaucoup sur X » ≠ « X provoque le défaut ». Un modèle
> peut s'appuyer massivement sur une variable qui n'est qu'un **reflet** d'autre
> chose — on le démontrera au § 5.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import sklearn

warnings.filterwarnings("ignore")  # confort de lecture ; à ne pas faire en prod

print("scikit-learn", sklearn.__version__, "| shap", shap.__version__)
# Versions de référence de ce notebook : scikit-learn 1.5.2, shap 0.52.0

## § 1 — Le dossier client, et deux modèles de référence

Le jeu de données est **synthétique et reproductible** (`donnees_aubrac.py`,
`random_state=42`). Son intérêt pédagogique : **on connaît la règle** qui
fabrique la cible. On pourra donc *vérifier* les explications au lieu de les
croire sur parole — un luxe que tu n'auras jamais sur des données réelles.

In [ ]:
from donnees_aubrac import (CIBLE, COLONNE_SENSIBLE, COLONNES_CATEGORIELLES,
                            COLONNES_NUMERIQUES, RANDOM_STATE, REGLE_GENERATRICE,
                            generer_dossiers)

dossiers = generer_dossiers(n=4000)
print(dossiers.shape, "| taux de défaut :", round(dossiers[CIBLE].mean(), 3))
dossiers.head(3)

In [ ]:
print(REGLE_GENERATRICE)

Lis bien la dernière ligne de la règle : `+ 0.450 * [sexe == F]`.

Les décisions passées de Crédit Aubrac étaient **biaisées** — et comme le label
`defaut` vient de l'historique, le biais est **dans la cible**. La colonne `sexe`
n'est évidemment **pas** donnée au modèle (c'est interdit, et de toute façon
elle n'est pas dans le fichier métier). On verra au § 5 que ça ne suffit pas.

Deux autres pièges volontaires sont posés dans les données :

| Variable | Ce qu'elle est vraiment |
|---|---|
| `reference_dossier` | un **numéro de dossier** : du bruit pur, aucune information |
| `interruption_carriere_mois` | une **variable proxy** : fortement corrélée au sexe |

In [ ]:
print(dossiers.groupby(COLONNE_SENSIBLE)[CIBLE].mean().round(3).to_string())
print("\nInterruption de carrière moyenne (mois) :")
print(dossiers.groupby(COLONNE_SENSIBLE)["interruption_carriere_mois"].mean().round(1).to_string())

### Trois références, pas une

Réflexe de la fiche-pattern : on ne juge jamais un modèle seul. On compare à un
**plancher trivial** (`DummyClassifier`) et à un **modèle interprétable par
construction** (régression logistique) avant de sortir la forêt aléatoire.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = dossiers.drop(columns=[CIBLE, COLONNE_SENSIBLE])   # le sexe ne va PAS au modèle
y = dossiers[CIBLE]

X_train, X_test, y_train, y_test, sexe_train, sexe_test = train_test_split(
    X, y, dossiers[COLONNE_SENSIBLE],
    test_size=0.25, random_state=RANDOM_STATE, stratify=y,
)

def preprocesseur() -> ColumnTransformer:
    return ColumnTransformer([
        ("num", StandardScaler(), COLONNES_NUMERIQUES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), COLONNES_CATEGORIELLES),
    ])

dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
logreg = Pipeline([("prep", preprocesseur()),
                   ("clf", LogisticRegression(max_iter=1000, class_weight="balanced",
                                              random_state=RANDOM_STATE))]).fit(X_train, y_train)
foret = Pipeline([("prep", preprocesseur()),
                  ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=10,
                                                 class_weight="balanced",
                                                 random_state=RANDOM_STATE, n_jobs=-1))]).fit(X_train, y_train)

for nom, modele in [("Dummy", dummy), ("LogReg", logreg), ("Forêt aléatoire", foret)]:
    pred = modele.predict(X_test)
    auc = roc_auc_score(y_test, modele.predict_proba(X_test)[:, 1])
    print(f"{nom:16s} F1-macro={f1_score(y_test, pred, average='macro'):.3f}"
          f"  F1-défaut={f1_score(y_test, pred, pos_label=1):.3f}  AUC={auc:.3f}")

> 🧭 **Résultat attendu** : Dummy ≈ 0.433, LogReg ≈ 0.734, forêt ≈ 0.749 de F1-macro.
>
> La forêt gagne **+0.015 de F1-macro** sur la régression logistique. Retiens ce
> chiffre : c'est le **prix payé en opacité** pour ce gain-là. Sur un dossier
> réglementaire (crédit, RH, santé), la question « est-ce que ça vaut le coup ? »
> se pose sérieusement — et ce n'est pas une question technique, c'est un
> arbitrage à porter devant le client.

Sur la régression logistique, l'explication est **déjà là**, sans rien installer :

In [ ]:
noms_features = logreg.named_steps["prep"].get_feature_names_out()
coefficients = pd.Series(logreg.named_steps["clf"].coef_[0], index=noms_features)
print("Coefficients (échelle standardisée), triés par |valeur| :")
print(coefficients.sort_values(key=abs, ascending=False).head(8).round(3).to_string())

Lecture : `taux_endettement` pèse ~6× plus que le 2ᵉ facteur, `nb_incidents_12m`
vient ensuite, `anciennete_emploi_mois` et `epargne_disponible` protègent (signe
négatif). C'est **exactement** la règle génératrice.

Deux remarques qui serviront plus loin :

- `interruption_carriere_mois` apparaît avec un coefficient positif (~0.24). Le
  modèle a donc appris à **pénaliser** cette variable, alors qu'elle n'est
  **pas** dans la règle génératrice. Garde ça en tête pour le § 5.
- `reference_dossier` a un coefficient non nul (~0.18) alors que c'est du bruit
  pur. **Un coefficient non nul ne prouve pas qu'une variable sert** : c'est du
  sur-ajustement de la variable sur le bruit. On tranchera au § 2.

## § 2 — L'importance globale, sans SHAP

Premier réflexe : **scikit-learn suffit**. Deux méthodes existent, elles ne se
valent pas du tout.

- `feature_importances_` (**importance par impureté**) : gratuite, calculée
  pendant l'entraînement… et **biaisée** vers les variables à nombreuses
  valeurs distinctes (numériques continues, identifiants). Elle mesure combien
  une variable a servi à *découper*, pas combien elle sert à *prédire*.
- `permutation_importance` : on mélange une colonne et on regarde **combien la
  métrique chute**, sur le jeu de test. Plus lent, mais ça répond à la vraie
  question. C'est la référence.

In [ ]:
from sklearn.inspection import permutation_importance

impurete = pd.Series(foret.named_steps["clf"].feature_importances_,
                     index=noms_features).sort_values(ascending=False)

resultat = permutation_importance(foret, X_test, y_test, n_repeats=10,
                                  random_state=RANDOM_STATE, scoring="f1_macro", n_jobs=-1)
permutation = pd.Series(resultat.importances_mean,
                        index=X_test.columns).sort_values(ascending=False)

print("--- par impureté (top 6) ---")
print(impurete.head(6).round(3).to_string())
print("\n--- par permutation, chute de F1-macro (top 6) ---")
print(permutation.head(6).round(4).to_string())

print("\nreference_dossier (bruit pur) :")
print("   impureté    :", round(impurete["num__reference_dossier"], 3))
print("   permutation :", round(permutation["reference_dossier"], 4))

> 🧭 **Le piège, en un chiffre.** `reference_dossier` — un **numéro de dossier**,
> zéro information — arrive **6ᵉ** en importance par impureté (~0.051), devant la
> moitié des vraies variables. Par permutation, sa contribution tombe à ~0.003 :
> mélanger la colonne ne change quasiment rien au F1.
>
> **Conséquence pratique** : ne colle jamais un graphique de `feature_importances_`
> dans une note client. Sur un jeu réel, tes identifiants, tes dates en timestamp
> et tes montants continus remonteront pour la même raison — et quelqu'un
> finira par bâtir une décision métier dessus.

Note aussi que la permutation confirme la lecture du § 1 : `taux_endettement`
écrase tout (≈ 0.18 de chute de F1), le reste est marginal — et
`interruption_carriere_mois` se glisse dans le top 4. On y revient.

## § 3 — L'explication locale avec SHAP

L'importance globale ne répond **pas** à la question du client : *pourquoi MON
dossier ?* Il faut une explication **locale**.

**L'idée de SHAP en une phrase** : on part de la prédiction moyenne du modèle
(la *valeur de base*), et on répartit l'écart entre cette moyenne et la
prédiction du dossier **entre les variables**, en évaluant la contribution de
chacune dans toutes les combinaisons possibles (les *valeurs de Shapley*, issues
de la théorie des jeux coopératifs).

D'où la propriété qui fait toute la valeur de SHAP : **l'additivité**.

```
valeur_de_base + somme(contributions) = prédiction du modèle, exactement
```

`TreeExplainer` calcule ça de façon **exacte et déterministe** pour les modèles
à base d'arbres (forêts, gradient boosting) — pas d'échantillonnage, pas d'aléa.
Pour les autres modèles il existe `KernelExplainer`, **beaucoup** plus lent
(§ 7).

In [ ]:
prep = foret.named_steps["prep"]
clf = foret.named_steps["clf"]

# SHAP s'applique au modèle final, donc aux données APRÈS transformation.
X_test_transforme = pd.DataFrame(prep.transform(X_test), columns=noms_features, index=X_test.index)

explainer = shap.TreeExplainer(clf)
# Laisse SHAP contrôler l'additivité : une explication incohérente doit échouer, pas être masquée.
valeurs = explainer.shap_values(X_test_transforme)
valeurs_defaut = valeurs[:, :, 1] if np.ndim(valeurs) == 3 else valeurs   # classe 1 = défaut
valeur_de_base = explainer.expected_value[1]

print("Une matrice de contributions, une par dossier et par variable :", valeurs_defaut.shape)
print("Valeur de base (probabilité moyenne prédite) :", round(valeur_de_base, 3))

### Rendre l'explication lisible

Petit détail qui n'en est pas un : SHAP travaille sur les colonnes
**transformées** — donc standardisées. Un waterfall affichant
`taux_endettement = 1.84` ne veut rien dire pour un conseiller clientèle.

On reconstruit donc une matrice d'**affichage** avec les valeurs métier
d'origine pour les colonnes numériques (les colonnes one-hot restent en 0/1).
Ce remapping est **à ta charge** — c'est le piège n°1 du § 7.

In [ ]:
donnees_affichage = X_test_transforme.copy()
donnees_affichage[[f"num__{c}" for c in COLONNES_NUMERIQUES]] = X_test[COLONNES_NUMERIQUES].to_numpy()

explication = shap.Explanation(
    values=valeurs_defaut,
    base_values=np.full(len(valeurs_defaut), valeur_de_base),
    data=donnees_affichage.to_numpy(),
    feature_names=[n.replace("num__", "").replace("cat__", "") for n in noms_features],
)

probabilites = foret.predict_proba(X_test)[:, 1]
i_refuse = int(np.argsort(probabilites)[int(0.97 * len(probabilites))])  # dossier à haut risque
i_accepte = int(np.argmin(probabilites))

print(f"Dossier refusé  : index {X_test.index[i_refuse]}, p(défaut) = {probabilites[i_refuse]:.3f}")
print(f"Dossier accepté : index {X_test.index[i_accepte]}, p(défaut) = {probabilites[i_accepte]:.3f}")

In [ ]:
shap.plots.waterfall(explication[i_refuse], max_display=8, show=False)
plt.title(f"Dossier refusé — p(défaut) = {probabilites[i_refuse]:.2f}")
plt.tight_layout(); plt.show()

In [ ]:
shap.plots.waterfall(explication[i_accepte], max_display=8, show=False)
plt.title(f"Dossier accepté — p(défaut) = {probabilites[i_accepte]:.2f}")
plt.tight_layout(); plt.show()

Lecture d'un waterfall : on part de `E[f(x)]` en bas (la valeur de base), chaque
barre pousse la prédiction vers le haut (rouge, aggrave) ou vers le bas (bleu,
protège), et on arrive à `f(x)` en haut — la probabilité réellement prédite.

Vérifions l'additivité, ce n'est pas un slogan :

In [ ]:
somme = valeurs_defaut[i_refuse].sum() + valeur_de_base
print(f"base {valeur_de_base:.3f} + somme des contributions {valeurs_defaut[i_refuse].sum():+.3f}"
      f" = {somme:.3f}   |   probabilité du modèle = {probabilites[i_refuse]:.3f}")

### 🎯 Le vrai livrable : la phrase pour le client

Un waterfall ne s'envoie pas à un demandeur de crédit. Le scoring de crédit est un cas où les textes
s'appliquent vraiment : l'AI Act le classe **haut risque** (Annexe III, 5 b —
évaluer la solvabilité d'une personne) et ouvre un **droit à explication** des
décisions individuelles (art. 86) ; le RGPD, **si la décision est
exclusivement automatisée** et produit un effet significatif — un refus de
crédit en est un, et la CJUE (*SCHUFA*, 2023) y inclut un score qui pèse de
façon déterminante —, impose l'art. 22 et une **information utile sur la
logique** de la décision (art. 13 à 15). Dans tous les cas, ce qu'on doit
pouvoir produire, c'est une **explication intelligible**, pas un graphe.

La cellule suivante transforme les contributions en phrase française. Lis le
gabarit, puis **modifie-le** : c'est ton exercice.

In [ ]:
def _fr(valeur: float) -> str:
    """Formate un nombre à la française : 35100.0 -> '35 100'."""
    return f"{valeur:,.0f}".replace(",", " ")

LIBELLES = {
    "taux_endettement": "un taux d'endettement de {v} %",
    "nb_incidents_12m": "{v} incident(s) de paiement sur 12 mois",
    "anciennete_emploi_mois": "une ancienneté professionnelle de {v} mois",
    "epargne_disponible": "une épargne disponible de {v} €",
    "montant_demande": "un montant demandé de {v} €",
    "duree_mois": "une durée de remboursement de {v} mois",
    "revenu_mensuel": "un revenu mensuel de {v} €",
    "interruption_carriere_mois": "une interruption de carrière de {v} mois",
}

def phrase_de_decision(i: int, n_facteurs: int = 3) -> str:
    """Traduit les contributions SHAP d'un dossier en une phrase pour le client."""
    contributions = pd.Series(valeurs_defaut[i], index=explication.feature_names)
    aggravants = contributions.sort_values(ascending=False).head(n_facteurs)
    facteurs = []
    for nom, contribution in aggravants.items():
        if contribution <= 0 or nom not in LIBELLES:
            continue
        facteurs.append(LIBELLES[nom].format(v=_fr(X_test.iloc[i][nom])))
    decision = "défavorable" if probabilites[i] >= 0.5 else "favorable"
    return (f"Avis {decision} (probabilité d'incident estimée à "
            f"{probabilites[i]:.0%}). Les éléments qui ont le plus pesé sont : "
            + ", ".join(facteurs) + ".")

print(phrase_de_decision(i_refuse))

> 🎯 **À toi.** Trois variantes à essayer, dans l'ordre de difficulté :
>
> 1. ajoute les facteurs **favorables** à la phrase (« ont joué en votre faveur : … ») ;
> 2. n'affiche un facteur que si sa contribution dépasse **5 % de l'écart total**
>    — un facteur à 0.002 n'intéresse personne ;
> 3. ajoute une phrase d'**actionnabilité** : de tes 3 facteurs, lesquels le
>    client peut-il réellement changer (le montant demandé, oui ; son ancienneté,
>    non) ? Une explication non actionnable est vécue comme une fin de non-recevoir.
>
> ⭐ **Pour aller plus loin** : ce que le client veut vraiment, c'est
> « qu'est-ce que je change pour passer ? ». Ça s'appelle une **explication
> contrefactuelle**, et ce n'est pas ce que SHAP calcule. Cherche `dice-ml`.

## § 4 — L'explication globale avec SHAP, et l'épreuve de vérité

En agrégeant les explications locales (`|contribution| moyenne`), on obtient une
vue globale — cohérente avec les explications individuelles, contrairement à
`feature_importances_`.

Et comme **on connaît la règle génératrice**, on peut faire ce qu'on ne pourra
jamais faire en vrai : vérifier.

In [ ]:
shap.plots.beeswarm(explication, max_display=9, show=False)
plt.tight_layout(); plt.show()

In [ ]:
global_shap = pd.Series(np.abs(valeurs_defaut).mean(0),
                        index=explication.feature_names).sort_values(ascending=False)
print(global_shap.head(8).round(4).to_string())

> 🧭 **Verdict, honnêtement.** SHAP retrouve bien le premier facteur de la règle
> (`taux_endettement`, très loin devant) et `nb_incidents_12m`. Mais
> `montant_demande` et `duree_mois` arrivent haut alors qu'ils **ne sont pas**
> dans la règle sous cette forme.
>
> Pourquoi ? Parce que `taux_endettement` a été **fabriqué** à partir de
> `montant_demande`, `duree_mois` et `revenu_mensuel`. Quand des variables sont
> corrélées, **SHAP répartit le crédit entre elles** : chacune peut se substituer
> partiellement aux autres, donc chacune reçoit sa part.
>
> Retiens la formulation juste : *« SHAP te dit sur quoi le modèle s'appuie, pas
> quelle variable est la cause. »* Sur des variables corrélées — c'est-à-dire
> presque toujours en vrai — la répartition est **une** répartition possible,
> pas *la* vérité.

Le beeswarm ajoute une information que le classement seul ne donne pas : le
**sens**. Une ligne = une variable, un point = un dossier, la couleur = la valeur
de la variable (rouge = élevée). Si les points rouges sont à droite, alors
« valeur élevée → pousse vers le défaut ».

## § 5 — L'explicabilité comme détecteur de proxy 🚨

Regarde la position de `interruption_carriere_mois` dans le beeswarm. Cette
variable n'est **pas** dans la règle génératrice du défaut. Et pourtant le modèle
s'en sert, et elle pousse dans un sens bien précis.

Souviens-toi : la colonne `sexe` n'a **jamais** été donnée au modèle. Croisons
malgré tout les contributions SHAP avec elle.

In [ ]:
i_proxy = list(explication.feature_names).index("interruption_carriere_mois")
contributions_proxy = pd.Series(valeurs_defaut[:, i_proxy], index=sexe_test.to_numpy())
print("Contribution SHAP moyenne de `interruption_carriere_mois` :")
print(contributions_proxy.groupby(level=0).mean().round(4).to_string())

predictions = foret.predict(X_test)
taux_refus = pd.Series(predictions, index=sexe_test.to_numpy()).groupby(level=0).mean()
print("\nTaux de refus prédit :", taux_refus.round(3).to_dict())
print("Ratio de refus F/H :", round(taux_refus["F"] / taux_refus["H"], 2))
taux_acceptation = 1 - taux_refus
print("Disparate impact (taux d'acceptation F/H) :",
      round(taux_acceptation["F"] / taux_acceptation["H"], 2))

> 🚨 **Ce qu'on vient de démontrer.** La contribution SHAP de
> `interruption_carriere_mois` est **positive en moyenne pour les femmes**
> (≈ +0.013) et **négative pour les hommes** (≈ −0.024). Autrement dit : le
> modèle a reconstruit un attribut sensible qu'on ne lui a jamais donné, et il
> s'en sert pour pénaliser : les femmes sont refusées **1.41 fois** plus souvent.
>
> Et pourtant, la règle des 4/5 ne voit **rien** : elle se calcule sur l'issue
> **favorable** (l'acceptation) — 0.647 / 0.750 ≈ **0.86**, au-dessus de 0.80. Le
> seuil n'est donc **pas une garantie** : ici, c'est le croisement SHAP × attribut
> sensible qui révèle le proxy. (On peut conclure parce qu'on **connaît la règle
> génératrice** ; sur des données réelles, ce serait un signal à investiguer.)
> Seuil conventionnel : règle des 4/5 vue en M2-B2.
>
> **Retirer la colonne sensible ne suffit pas.** C'est le point à retenir de ce
> notebook si tu n'en retiens qu'un. L'explicabilité est ici un **outil d'audit
> éthique** (C2), pas seulement un outil de communication.

Question suivante, et c'est celle du client : **est-ce que ça coûte cher de
corriger ?**

In [ ]:
def entrainer_sans(colonnes: list[str]) -> tuple[float, float, float]:
    """Réentraîne la forêt sans certaines colonnes ; renvoie (F1-macro, ratio de refus F/H, DI d'acceptation F/H)."""
    restantes_num = [c for c in COLONNES_NUMERIQUES if c not in colonnes]
    modele = Pipeline([
        ("prep", ColumnTransformer([
            ("num", StandardScaler(), restantes_num),
            ("cat", OneHotEncoder(handle_unknown="ignore"), COLONNES_CATEGORIELLES)])),
        ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=10,
                                       class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)),
    ]).fit(X_train.drop(columns=colonnes), y_train)
    pred = modele.predict(X_test.drop(columns=colonnes))
    taux = pd.Series(pred, index=sexe_test.to_numpy()).groupby(level=0).mean()
    return f1_score(y_test, pred, average="macro"), taux["F"] / taux["H"], (1 - taux["F"]) / (1 - taux["H"])

for libelle, colonnes in [("modèle complet", []),
                          ("sans le proxy", ["interruption_carriere_mois"])]:
    f1, ratio, di = entrainer_sans(colonnes)
    print(f"{libelle:18s} F1-macro = {f1:.3f}   ratio de refus F/H = {ratio:.2f}   DI acceptation = {di:.2f}")

> 🧭 **Le chiffre qui clôt le débat** : retirer le proxy coûte **0.002 de
> F1-macro** (0.749 → 0.747, c'est du bruit) et fait tomber le ratio de refus de
> **1.41 à 1.16** (DI d'acceptation 0.86 → 0.94). Il n'y a pas d'arbitrage performance/équité ici — il y a une
> variable qu'on gardait par négligence.
>
> Note bien que le ratio ne tombe **pas à 1.00**. Le reste de l'écart vient du
> **label lui-même** : l'historique de décisions était biaisé. Aucune méthode
> d'explicabilité ne répare une cible biaisée — ça, c'est un sujet de collecte
> et de gouvernance (C1/C2), pas de modélisation.
>
> 🎯 **À toi** : relance `entrainer_sans` en retirant aussi `reference_dossier`.
> Le F1 bouge-t-il ? Que conclus-tu sur le fait de garder des colonnes « au cas où » ?

## § 6 — Et LIME ?

LIME répond à la même question que SHAP local, autrement : il **perturbe** le
dossier des milliers de fois autour de sa valeur, observe les prédictions, et
ajuste un **modèle linéaire local** sur ce nuage. L'explication, c'est ce modèle
linéaire.

Conséquence directe : le résultat dépend de l'**échantillonnage**. Regardons.

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

explainer_lime = LimeTabularExplainer(
    np.asarray(prep.transform(X_train)),
    feature_names=list(noms_features), class_names=["sain", "défaut"], mode="classification",
)

for n_echantillons in (100, 5000):
    print(f"--- num_samples = {n_echantillons} ---")
    for graine in range(3):
        np.random.seed(graine)
        e = explainer_lime.explain_instance(
            np.asarray(X_test_transforme)[i_refuse], clf.predict_proba,
            num_features=3, num_samples=n_echantillons)
        print("   ", [(f, round(p, 3)) for f, p in e.as_list()])

> 🧭 **Lecture.** À 100 échantillons, l'**ordre** des variables 2 et 3 change
> d'une exécution à l'autre. À 5000, c'est stable. LIME n'est donc pas
> « instable » au sens de faux — il est **approximatif et paramétré** : tu dois
> choisir un budget d'échantillonnage, et le justifier.
>
> Deux autres différences pratiques sautent aux yeux ci-dessus :
> - LIME te rend des **seuils** (`taux_endettement > 0.54`) — pratique à
>   raconter… sauf que ce sont des seuils sur des valeurs **standardisées**,
>   illisibles tels quels ;
> - LIME n'est **pas additif** : ses poids ne se recomposent pas en la
>   probabilité prédite. Tu ne peux pas écrire « base + contributions = décision ».

### Le verdict, à recopier dans ta grille C4

| Situation | Ce que tu utilises |
|---|---|
| Décision réglementée, modèle simple suffisant | **Modèle interprétable par construction** (logistique, arbre court). Zéro dépendance, explication exacte. |
| Modèle à base d'arbres, besoin local **et** global | **SHAP `TreeExplainer`** : exact, déterministe, additif, rapide (~4 s pour 1000 dossiers ici). |
| Modèle non-arbre (SVM, réseau, boîte noire d'API) | **`KernelExplainer` ou LIME** — les deux approximent ; LIME est souvent bien plus rapide. |
| Texte ou image | LIME et SHAP ont des variantes dédiées ; regarde d'abord les cartes d'attention / Grad-CAM. |
| Besoin d'un « que dois-je changer ? » | **Ni l'un ni l'autre** → explication contrefactuelle (`dice-ml`). |

## § 7 — Pièges fréquents

| Piège | Conséquence |
|---|---|
| Coller un graphe `feature_importances_` dans une note client | Les identifiants et variables continues remontent artificiellement (démontré au § 2) et fondent une décision métier fausse |
| Expliquer sans remapper les features transformées | Le client lit `taux_endettement = 1.84` (standardisé) ou `cat__type_contrat_CDD` : illisible, donc inutilisable |
| Lire SHAP comme de la causalité | « Réduire son interruption de carrière ferait baisser son risque » — faux, et discriminatoire |
| Ignorer la corrélation entre variables | Le crédit est réparti entre variables jumelles ; deux features « moyennes » peuvent être une seule cause (§ 4) |
| Sortir `KernelExplainer` sur un modèle à arbres | Des minutes de calcul et une approximation, là où `TreeExplainer` donne l'exact en secondes |
| Croire qu'expliquer = se conformer | L'explication documente la décision ; elle ne la rend ni juste, ni légale (§ 5) |

| Symptôme | Cause probable |
|---|---|
| `shap_values` renvoie une forme `(n, k, 2)` inattendue | Classifieur binaire : la 3ᵉ dimension est la classe. Prends `[:, :, 1]` |
| Erreur d'additivité (`Additivity check failed`) | Tu passes au `TreeExplainer` des données **non transformées**, ou pas les mêmes colonnes qu'à l'entraînement |
| Le waterfall affiche des valeurs entre −3 et +3 | Tu affiches les données standardisées : reconstruis une matrice d'affichage métier |
| SHAP tourne des minutes sur une forêt | Tu as pris `KernelExplainer` (ou `shap.Explainer` sans préciser) au lieu de `TreeExplainer` |
| Deux exécutions de LIME donnent deux classements | `num_samples` trop faible — c'est le fonctionnement normal, pas un bug |
| Une variable interdite ressort en tête | Tu tiens probablement un **proxy** : croise ses contributions avec l'attribut sensible (§ 5) |

## ✅ Checklist de sortie

- [ ] Je sais dire en 2 minutes la différence entre explication **globale** et **locale**.
- [ ] Je sais pourquoi `permutation_importance` est préférable à `feature_importances_`, avec un chiffre à l'appui.
- [ ] J'ai produit un waterfall SHAP et vérifié que `base + contributions = prédiction`.
- [ ] J'ai rédigé une phrase de refus lisible par un client, et je sais laquelle de ses composantes est actionnable.
- [ ] Je sais expliquer pourquoi retirer la colonne sensible ne suffit pas à retirer le biais.
- [ ] Je sais dire dans quel cas SHAP est du **sur-engineering** (et je peux nommer le modèle à préférer).
- [ ] Je peux remplir la ligne « explicabilité » de `grille_decision_C4.md` avec des arguments, pas des impressions.

## Pour aller plus loin

- Fiche du parcours : `fiche_interpretabilite_xai.pdf` (feature importance, SHAP, LIME, PDP).
- Documentation SHAP : <https://shap.readthedocs.io/en/latest/>
- `sklearn.inspection` (permutation importance, PDP, ICE) : <https://scikit-learn.org/stable/modules/partial_dependence.html>
- Christoph Molnar, *Interpretable Machine Learning* (gratuit en ligne) : <https://christophm.github.io/interpretable-ml-book/> — chapitres 8 (méthodes agnostiques) et 9 (SHAP).
- Suite directe : **`02_explicabilite_sur_ton_modele_TODO.ipynb`** — le même geste, sur un vrai modèle à toi, sans guidage.